In [2]:
import torch
from transformers import AutoModel, AutoTokenizer
from vllm import LLM, SamplingParams



INFO 02-24 14:55:54 __init__.py:183] Automatically detected platform cuda.


In [24]:

model = LLM(model='intfloat/e5-small-v2')

INFO 02-24 15:07:28 config.py:2314] Downcasting torch.float32 to torch.float16.
INFO 02-24 15:07:28 config.py:520] This model supports multiple tasks: {'classify', 'embed', 'reward', 'score'}. Defaulting to 'embed'.
INFO 02-24 15:07:29 llm_engine.py:232] Initializing an LLM engine (v0.7.0) with config: model='intfloat/e5-small-v2', speculative_config=None, tokenizer='intfloat/e5-small-v2', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 02-24 15:07:32 model_runner.py:1115] Loading model weights took 0.0639 GB


In [25]:

e1 = model.encode('query: x : ℕ\nh₀ : ↑x + 4 / 100 * ↑x = 598\n⊢ 100 * x = 100 * 575')[0].outputs.data.cpu().numpy()


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 48.36it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


In [29]:
e2 = e2[0].detach().numpy()


In [35]:
e2 - e1

array([ 8.12709332e-05, -2.36034393e-05,  6.46784902e-05, -2.71545723e-05,
       -1.49151310e-05, -5.56744635e-06, -3.06144357e-05, -9.22977924e-05,
        3.25087458e-05, -1.09642744e-04,  5.98989427e-05,  1.56968832e-04,
       -2.96942890e-05, -7.84564763e-05,  9.60193574e-06,  3.84915620e-05,
       -3.15159559e-06, -8.04290175e-05,  9.37134027e-05, -2.80141830e-06,
       -1.55270100e-05,  2.65687704e-05,  1.56313181e-05,  3.98382545e-05,
        8.28392804e-05,  2.54386105e-05, -4.29898500e-05, -1.28764659e-04,
        2.41883099e-05, -1.82643533e-04, -9.92855057e-05,  6.43357635e-06,
        5.94370067e-05, -4.30345535e-05, -1.13561749e-04, -4.31872904e-05,
       -1.38346106e-04, -8.14273953e-05, -2.82097608e-06, -1.01476908e-05,
        1.50799751e-05,  1.26697123e-05,  6.19888306e-06, -4.56422567e-05,
        1.48992985e-05,  7.67409801e-06,  2.34097242e-05, -6.46188855e-05,
       -2.03698874e-05,  5.07999212e-05, -5.52274287e-05,  1.76820904e-05,
        9.44174826e-05,  

In [3]:

model = LLM(model='internlm/internlm2_5-step-prover-critic', trust_remote_code=True, task='reward')
chat_1 = [
    {"role": "user", "content": "Which state is closer to 'no goals'?"},
    {"role": "assistant", "content": "no goals"}
]

model.reward(chat_1)



INFO 02-17 15:47:58 config.py:134] Replacing legacy 'type' key with 'rope_type'
WARNING 02-17 15:48:09 arg_utils.py:1117] The model has a long context length (65536). This may cause OOM errors during the initial memory profiling phase, or result in low performance due to small KV cache space. Consider setting --max-model-len to a smaller value.
INFO 02-17 15:48:09 llm_engine.py:232] Initializing an LLM engine (v0.7.0) with config: model='internlm/internlm2_5-step-prover-critic', speculative_config=None, tokenizer='internlm/internlm2_5-step-prover-critic', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=65536, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend=

Loading pt checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 02-17 15:48:17 model_runner.py:1115] Loading model weights took 3.1819 GB


AttributeError: 'LLM' object has no attribute 'reward'

In [25]:

model = AutoModel.from_pretrained(
    "internlm/internlm2_5-step-prover-critic",
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("internlm/internlm2_5-step-prover-critic", trust_remote_code=True)

chat_1 = [
    {"role": "user", "content": "Which state is closer to 'no goals'?"},
    {"role": "assistant", "content": "no goals"}
]
chat_2 = [
    {"role": "user", "content": "Which state is closer to 'no goals'?"},
    {"role": "assistant", "content": "x : ℕ\nh₀ : ↑x + 4 / 100 * ↑x = 598\n⊢ 100 * x = 100 * 575"}
]

score1 = model.get_score(tokenizer, chat_1)
score2 = model.get_score(tokenizer, chat_2)
print("score1: ", score1)
print("score2: ", score2)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

score1:  16.5
score2:  1.595703125


In [ ]:

model = AutoModel.from_pretrained(
    "internlm/internlm2_5-step-prover",
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("internlm/internlm2_5-step-prover", trust_remote_code=True)

In [2]:
llm = LLM(model='internlm/internlm2_5-step-prover', trust_remote_code=True, tensor_parallel_size=2)


INFO 02-14 15:30:04 config.py:134] Replacing legacy 'type' key with 'rope_type'
INFO 02-14 15:30:14 config.py:520] This model supports multiple tasks: {'classify', 'generate', 'reward', 'embed', 'score'}. Defaulting to 'generate'.
INFO 02-14 15:30:14 config.py:1328] Defaulting to use mp for distributed inference
INFO 02-14 15:30:14 llm_engine.py:232] Initializing an LLM engine (v0.7.0) with config: model='internlm/internlm2_5-step-prover', speculative_config=None, tokenizer='internlm/internlm2_5-step-prover', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityC

Loading pt checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]


INFO 02-14 15:30:42 model_runner.py:1115] Loading model weights took 7.2095 GB
(VllmWorkerProcess pid=73005) INFO 02-14 15:30:42 model_runner.py:1115] Loading model weights took 7.2095 GB
(VllmWorkerProcess pid=73005) INFO 02-14 15:30:47 worker.py:266] Memory profiling takes 4.68 seconds
(VllmWorkerProcess pid=73005) INFO 02-14 15:30:47 worker.py:266] the current vLLM instance can use total_gpu_memory (47.29GiB) x gpu_memory_utilization (0.90) = 42.56GiB
(VllmWorkerProcess pid=73005) INFO 02-14 15:30:47 worker.py:266] model weights take 7.21GiB; non_torch_memory takes 0.38GiB; PyTorch activation peak memory takes 0.65GiB; the rest of the memory reserved for KV Cache is 34.32GiB.
INFO 02-14 15:30:47 worker.py:266] Memory profiling takes 4.43 seconds
INFO 02-14 15:30:47 worker.py:266] the current vLLM instance can use total_gpu_memory (47.32GiB) x gpu_memory_utilization (0.90) = 42.59GiB
INFO 02-14 15:30:47 worker.py:266] model weights take 7.21GiB; non_torch_memory takes 0.41GiB; PyTorc

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:24<00:00,  1.41it/s]

INFO 02-14 15:31:14 custom_all_reduce.py:224] Registering 2275 cuda graph addresses
(VllmWorkerProcess pid=73005) INFO 02-14 15:31:14 custom_all_reduce.py:224] Registering 2275 cuda graph addresses
(VllmWorkerProcess pid=73005) INFO 02-14 15:31:15 model_runner.py:1558] Graph capturing finished in 25 secs, took 0.43 GiB
INFO 02-14 15:31:15 model_runner.py:1558] Graph capturing finished in 25 secs, took 0.43 GiB
INFO 02-14 15:31:15 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 32.08 seconds


In [3]:
prompt = [f"---\nNAME: {'square_sub_one_divisible_eight'}\n\n---\nPROOF_BEFORE: {'rw [h, pow_two]'}\n\n---\nSTATE_BEFORE: 'm n : N\nh : n = 2 * m + 1\n⊢ 8 | n * n - 1'\n\n---\nTACTIC: "]


In [49]:
sampling_params = SamplingParams(n=32, temperature=0.7, stop_token_ids=[92542], best_of=32, logprobs=0)#, top_p=0.95)
out = llm.generate(prompt, sampling_params)



Processed prompts:   3%|▎         | 1/32 [00:00<00:20,  1.54it/s, est. speed input: 103.16 toks/s, output: 538.87 toks/s]


In [51]:

[(i.text.strip(), i.cumulative_logprob) for i in out[0].outputs]

[('rw [h, mul_add, mul_one]', -1.960043552557181),
 ('rw [h, mul_add, mul_one]', -1.960043552557181),
 ('rw [h, mul_add, mul_one]', -1.960043552557181),
 ('rw [h]', -2.105288189071871),
 ('rw [h]', -2.105288189071871),
 ('rw [h]', -2.105288189071871),
 ('rw [h, mul_add, mul_one, add_comm]', -2.1188550177248544),
 ('rw [h, mul_add, mul_one, add_comm]', -2.1188550177248544),
 ('rw [h, Nat.mul_add, Nat.mul_one]', -2.391726188005123),
 ('rw [h, Nat.mul_add, Nat.mul_one]', -2.391726188005123),
 ('rw [h, Nat.mul_add, Nat.mul_one]', -2.391726188005123),
 ('rw [h, Nat.mul_add, Nat.mul_one]', -2.391726188005123),
 ('rw [h, pow_two]', -2.799261191441474),
 ('rw [h, pow_two]', -2.799261191441474),
 ('rw [h, pow_two]', -2.799261191441474),
 ('rw [h, pow_two]', -2.799261191441474),
 ('simp [h, Nat.mul_mod, Nat.add_mod, Nat.mod_mod]', -2.840807825133652),
 ('ring_nf', -3.2021668000525096),
 ('ring_nf', -3.2021668000525096),
 ('ring_nf', -3.2021668000525096),
 ('rw [h, add_mul, one_mul]', -3.72579727

In [45]:

out[0].outputs[0]


CompletionOutput(index=0, text='rw [h, mul_add, mul_one]\n\n', token_ids=(31948, 640, 280, 328, 15734, 3054, 328, 15734, 11817, 2690, 92542), cumulative_logprob=-1.955109274473216, logprobs=[{31948: Logprob(logprob=-0.31204405426979065, rank=1, decoded_token='rw')}, {640: Logprob(logprob=-8.010543388081715e-05, rank=1, decoded_token=' [')}, {280: Logprob(logprob=-0.0015279296785593033, rank=1, decoded_token='h')}, {328: Logprob(logprob=-0.184807687997818, rank=1, decoded_token=',')}, {15734: Logprob(logprob=-0.6534253358840942, rank=1, decoded_token=' mul')}, {3054: Logprob(logprob=-0.019588593393564224, rank=1, decoded_token='_add')}, {328: Logprob(logprob=-0.02790931798517704, rank=1, decoded_token=',')}, {15734: Logprob(logprob=-0.05755041539669037, rank=1, decoded_token=' mul')}, {11817: Logprob(logprob=-0.004202107898890972, rank=1, decoded_token='_one')}, {2690: Logprob(logprob=-0.6936477422714233, rank=1, decoded_token=']\n\n'), 328: Logprob(logprob=-0.6936477422714233, rank=1, 

In [ ]:
print(prompt)

In [ ]:

tokenized_state = tokenizer(
            prompt,
            padding="longest",
            max_length=2000,
            truncation=True,
            return_tensors="pt",
        )

state_ids = tokenized_state.input_ids.cuda()

In [ ]:
out = model.generate(
            input_ids=state_ids,
            # max_new_tokens=4,
            num_return_sequences=64,
            do_sample=True,
            output_scores=True,
            return_dict_in_generate=True,
            temperature=0.7
        )


# Return the output.
raw_output_text = tokenizer.batch_decode(
    out.sequences, skip_special_tokens=False
)


In [ ]:

raw_output_text